# 📋 Gabarito — TODOs: Algoritmos de Diferença Temporal

> ⚠️ **Este documento é para consulta após a tentativa individual.**
> Tente resolver cada TODO antes de abrir o gabarito correspondente.

---

## TODO 1/4 — Atualização TD(0)

**Fórmula de referência:**
$$V(s) \leftarrow V(s) + \alpha \cdot \big[\underbrace{r + \gamma V(s')}_{\text{td\_target}} - V(s)\big]$$

### ✅ Solução

```python
td_target = reward + gamma * V[next_state]
td_error  = td_target - V[state]
V[state]  = V[state] + alpha * td_error
```

> A terceira linha pode ser escrita de forma equivalente como `V[state] += alpha * td_error`.

### Por que funciona?

| Variável | Representa | Valor típico |
|----------|-----------|--------------|
| `reward` | $r$ — recompensa imediata observada | `−0.04` (living reward) ou `+1` (terminal) |
| `gamma * V[next_state]` | $\gamma V(s')$ — valor futuro descontado | estimativa atual, **bootstrap** |
| `td_target` | $r + \gamma V(s')$ — o "quanto valeria daqui" | alvo que queremos atingir |
| `td_error` | diferença entre alvo e estimativa atual | sinal de erro que direciona a atualização |
| `alpha * td_error` | passo de ajuste | proporcional ao erro e à taxa de aprendizado |

**Ponto-chave:** O nome *bootstrap* vem do fato de `V[next_state]` ser uma **estimativa**, não um valor real.
TD(0) atualiza V(s) usando outra estimativa — é como "puxar a si mesmo pelos próprios cadarços".

---
## TODO 2/4 — Atualização SARSA (on-policy)

**Fórmula de referência:**
$$Q(s,a) \leftarrow Q(s,a) + \alpha \cdot \big[r + \gamma \underbrace{Q(s', a')}_{\text{ação REAL}} - Q(s,a)\big]$$

### ✅ Solução

```python
td_target = reward + gamma * Q[ns_idx, next_action_idx]
td_error  = td_target - Q[s_idx, action_idx]
Q[s_idx, action_idx] += alpha * td_error
```

### Por que `next_action_idx` e não `np.max`?

SARSA é **on-policy**: a política que *gera* a experiência é a mesma que está sendo *aprimorada*.

```
Sequência SARSA num passo:
  1. Estou em s,  escolhi a  (ε-greedy)
  2. Tomei a,  observei r e s'
  3. Já escolhi a' para s'  (ε-greedy) ← ANTES de atualizar Q
  4. Atualizo Q(s,a) usando Q(s', a')  ← ação que REALMENTE tomarei
  5. Avanço: s ← s', a ← a'
```

Se eu usasse `np.max(Q[ns_idx])` estaria aprendendo sobre a **melhor ação possível**, não sobre
a ação que minha política $\varepsilon$-greedy vai de fato tomar. Isso transformaria SARSA em Q-Learning.

> **Analogia:** SARSA avalia o seu comportamento real (incluindo erros de exploração).
> É como avaliar um motorista considerando que ele vai se distrair 10% do tempo.

---
## TODO 3/4 — Atualização Q-Learning (off-policy)

**Fórmula de referência:**
$$Q(s,a) \leftarrow Q(s,a) + \alpha \cdot \big[r + \gamma \underbrace{\max_{a'} Q(s', a')}_{\text{melhor possível}} - Q(s,a)\big]$$

### ✅ Solução

```python
max_next_q           = np.max(Q[ns_idx])
td_target            = reward + gamma * max_next_q
td_error             = td_target - Q[s_idx, action_idx]
Q[s_idx, action_idx] += alpha * td_error
```

### Comparação direta com SARSA

```
SARSA      →  td_target = reward + gamma * Q[ns_idx, next_action_idx]
Q-Learning →  td_target = reward + gamma * np.max(Q[ns_idx])
                                            ^^^^^^^^^^^^^^^^^^
                                            única linha diferente
```

Q-Learning é **off-policy**: aprende a política ótima $Q^*$ **independentemente** de como explora.
A política $\varepsilon$-greedy é usada apenas para *gerar* experiência; o alvo sempre usa o máximo.

| | SARSA | Q-Learning |
|--|-------|-----------|
| Política aprendida | $Q^\pi$ (on-policy) | $Q^*$ (ótima) |
| Alvo TD usa | Ação $a'$ escolhida por $\varepsilon$-greedy | $\max_{a'} Q(s',a')$ |
| No CliffWorld | Aprende rota segura | Aprende rota ótima (beira do precipício) |

> `np.max(Q[ns_idx])` percorre todos os valores de Q para o próximo estado
> e retorna o maior — sem precisar saber qual ação o produz.

---
## TODO 4/4 — Alvo do Expected SARSA

**Fórmula de referência:**
$$E_{\pi}[Q(s',a')] = \sum_a \pi(a|s') \cdot Q(s', a)$$

Sob política $\varepsilon$-greedy com $n$ ações disponíveis:

$$\pi(a|s') = \begin{cases} \dfrac{\varepsilon}{n} + (1-\varepsilon) & \text{se } a = \arg\max_{a} Q(s',a) \\ \dfrac{\varepsilon}{n} & \text{caso contrário} \end{cases}$$

### ✅ Solução

```python
q_prox  = Q[ns_idx]                            # Q(s', :) — todos os Q-values do próximo estado

# Passo 1 — probabilidade base para TODAS as ações
melhor  = np.argmax(q_prox)                     # índice da ação greedy
probs   = np.ones(n_actions) * (epsilon / n_actions)  # ε / n para todas

# Passo 2 — ação greedy recebe probabilidade extra
probs[melhor] += (1 - epsilon)                  # += (1 - ε)

# Passo 3 — expectativa = produto escalar
exp_q   = np.dot(probs, q_prox)                 # Σ π(a|s') · Q(s', a)
```

### Verificação: as probabilidades somam 1?

$$\sum_a \pi(a|s') = (n-1) \cdot \frac{\varepsilon}{n} + \left(\frac{\varepsilon}{n} + 1 - \varepsilon\right) = \frac{\varepsilon(n-1)}{n} + \frac{\varepsilon}{n} + 1 - \varepsilon = \varepsilon + 1 - \varepsilon = 1 \checkmark$$

### Por que Expected SARSA reduz variância?

| Algoritmo | Alvo TD | Variância |
|-----------|---------|-----------|
| SARSA | $Q(s', a')$ — **uma** ação amostrada | Alta: depende da ação sorteada |
| Q-Learning | $\max_a Q(s', a)$ — determinístico | Baixa, mas enviesado (off-policy) |
| **Expected SARSA** | $E_\pi[Q(s', a)]$ — **média** sobre todas | **Menor** que SARSA, sem viés off-policy |

> Expected SARSA é estritamente melhor que SARSA em termos de variância do gradiente.
> Quando $\varepsilon \rightarrow 0$, Expected SARSA converge para Q-Learning (o máximo vira a expectativa).

---

## Resumo das 4 equações lado a lado

```python
# TD(0)
td_target = reward + gamma * V[next_state]
V[state] += alpha * (td_target - V[state])

# SARSA  (on-policy)
td_target = reward + gamma * Q[ns_idx, next_action_idx]   # ação REAL
Q[s_idx, action_idx] += alpha * (td_target - Q[s_idx, action_idx])

# Q-Learning  (off-policy)
td_target = reward + gamma * np.max(Q[ns_idx])             # ação MÁXIMA
Q[s_idx, action_idx] += alpha * (td_target - Q[s_idx, action_idx])

# Expected SARSA
probs         = np.ones(n_actions) * (epsilon / n_actions)
probs[np.argmax(Q[ns_idx])] += (1 - epsilon)
td_target     = reward + gamma * np.dot(probs, Q[ns_idx])  # ESPERANÇA
Q[s_idx, action_idx] += alpha * (td_target - Q[s_idx, action_idx])
```

> **Toda a diferença está em como calcular `td_target`.**
> O restante da estrutura — episódio, transição, atualização incremental — é idêntico.